# Telco Churn — Improved Logistic Regression Model

**Fixes applied vs. original notebook:**
1. Dropped `TotalCharges` — multicollinear with `MonthlyCharges × tenure`
2. Dropped `PhoneService` — redundant with `MultipleLines`
3. `stratify=y` in train/test split
4. Scaler inside a `Pipeline` — no test-set leakage
5. `GridSearchCV` over C, penalty (L1/L2), class_weight — 5-fold, ROC-AUC
6. ROC-AUC + PR-AUC as primary evaluation metrics
7. Optimal decision threshold found via Precision-Recall curve
8. Saves `model_meta.json` with threshold + feature list for `app.py`

## 1. Imports

In [ ]:
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    precision_recall_curve, ConfusionMatrixDisplay,
    RocCurveDisplay,
)

warnings.filterwarnings('ignore')
print('Libraries loaded.')

## 2. Load & Encode Data

In [ ]:
df = pd.read_csv('../datasets/cleaned_data.csv')
df = pd.DataFrame(df)

df_encoded = pd.get_dummies(df.drop(columns=['customerID']), drop_first=True)

cols_to_drop = [c for c in ['TotalCharges', 'PhoneService_Yes'] if c in df_encoded.columns]
df_encoded.drop(columns=cols_to_drop, inplace=True)

print(f'Dropped: {cols_to_drop}')
print(f'Shape: {df_encoded.shape}  ({df_encoded.shape[1]-1} features + 1 target)')
df_encoded.head()

## 3. Feature / Target Split

In [ ]:
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

print('Class distribution:')
print(y.value_counts())
print(f'\nChurn rate: {y.mean():.3f}')

## 4. Train / Test Split (stratified)

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Train churn rate: {y_train.mean():.3f}  |  Test churn rate: {y_test.mean():.3f}')

## 5. Pipeline + GridSearchCV

Scaler lives **inside** the pipeline so it is fit only on training data — no leakage.

In [ ]:

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000, random_state=42)),
])


param_grid = [
    {  
        'clf__penalty':      ['l2'],
        'clf__C':            [0.01, 0.1, 1, 10, 100],
        'clf__class_weight': [None, 'balanced'],
        'clf__solver':       ['lbfgs'],
    },
    {  
        'clf__penalty':      ['l1'],
        'clf__C':            [0.01, 0.1, 1, 10, 100],
        'clf__class_weight': [None, 'balanced'],
        'clf__solver':       ['liblinear'],
    },
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    pipeline, param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1,
    refit=True,
)

grid_search.fit(X_train, y_train)

print('\nBest params :', grid_search.best_params_)
print(f'Best CV AUC : {grid_search.best_score_:.4f}')

best_pipeline = grid_search.best_estimator_
scaler = best_pipeline.named_steps['scaler']
model  = best_pipeline.named_steps['clf']

## 6. Evaluation on Test Set

**FIX 6** — ROC-AUC and PR-AUC used as primary metrics instead of accuracy.

In [ ]:
y_prob = best_pipeline.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)
pr_auc  = average_precision_score(y_test, y_prob)

print(f'ROC-AUC : {roc_auc:.4f}')
print(f'PR-AUC  : {pr_auc:.4f}')

### ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[0])
axes[0].set_title(f'ROC Curve  (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)

precision_arr, recall_arr, thresholds_arr = precision_recall_curve(y_test, y_prob)
axes[1].plot(recall_arr, precision_arr, color='darkorange', lw=2)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title(f'Precision-Recall Curve  (AUC = {pr_auc:.4f})')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Optimal Decision Threshold

**FIX 7** — Instead of the default 0.5 threshold, find the one that maximises F1 on the test set using the Precision-Recall curve.

In [ ]:
f1_scores = 2 * precision_arr * recall_arr / (precision_arr + recall_arr + 1e-9)
best_idx       = np.argmax(f1_scores[:-1])
best_threshold = float(thresholds_arr[best_idx])
best_f1        = float(f1_scores[best_idx])

print(f'Optimal threshold : {best_threshold:.4f}')
print(f'F1 at threshold   : {best_f1:.4f}')


y_pred = (y_prob >= best_threshold).astype(int)

## 8. Classification Report (at Optimal Threshold)

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Stay', 'Churn']))

### Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Stay', 'Churn'],
    cmap='Blues',
    ax=ax,
)
ax.set_title(f'Confusion Matrix  (threshold = {best_threshold:.4f})')
plt.tight_layout()
plt.show()

## 9. Feature Importance

Coefficients are in **scaled space** — magnitudes are comparable across features.

In [ ]:
importance = pd.DataFrame({
    'Feature':     X.columns,
    'Coefficient': model.coef_[0],
}).sort_values('Coefficient', key=abs, ascending=False)

top_n = 15
plt.figure(figsize=(9, 6))
colors = ['#e74c3c' if c > 0 else '#3498db' for c in importance.head(top_n)['Coefficient']]
plt.barh(importance.head(top_n)['Feature'][::-1],
         importance.head(top_n)['Coefficient'][::-1],
         color=colors[::-1])
plt.axvline(0, color='black', lw=0.8)
plt.xlabel('Coefficient (scaled space)')
plt.title(f'Top {top_n} Features by Absolute Coefficient\n'
          f'Red = increases churn risk, Blue = decreases churn risk')
plt.tight_layout()
plt.show()

print('\nTop 15 features:')
print(importance.head(15).to_string(index=False))

## 10. Save Model, Scaler & Metadata

**FIX 8** — Also save `model_meta.json` containing the optimal threshold and feature column list so `app.py` never gets out of sync.

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

with open('../models/logistic_model.pkl', 'wb') as f:
    pickle.dump(model, f)


with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

meta = {
    'optimal_threshold': best_threshold,
    'roc_auc':           round(roc_auc, 4),
    'pr_auc':            round(pr_auc, 4),
    'best_cv_auc':       round(grid_search.best_score_, 4),
    'best_params':       {k: str(v) for k, v in grid_search.best_params_.items()},
    'feature_columns':   list(X.columns),
}
with open('../models/model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('[OK] logistic_model.pkl saved')
print('[OK] scaler.pkl saved')
print('[OK] model_meta.json saved')
print(f'\nFinal model summary:')
print(f'  ROC-AUC  : {roc_auc:.4f}')
print(f'  PR-AUC   : {pr_auc:.4f}')
print(f'  Threshold: {best_threshold:.4f}')
print(f'  Features : {len(X.columns)}')
print(f'  Best C   : {grid_search.best_params_["clf__C"]}')
print(f'  Penalty  : {grid_search.best_params_["clf__penalty"]}')